# 09 — Monitoring and retraining signals

A deployed forecast fails quietly: the model keeps emitting plausible numbers
while its accuracy decays or its inputs drift. Nobody notices until rosters
have been wrong for a month. This notebook demonstrates the monitoring layer
(`clinic_forecast.monitoring`) that turns "is the model still OK?" into a
table an operations analyst can read in one minute.

Design principles:

- **Plain thresholds, not anomaly black boxes.** Every alert is
  "value X crossed threshold Y", with thresholds documented in
  `configs/monitoring.yaml`. An alerting system nobody understands gets
  ignored.
- **Monitor inputs, not just accuracy.** Demand and marketing distribution
  shifts predict accuracy problems *before* enough scored days accumulate to
  prove them.
- **Compare against training-time quality.** A clinic at 25% WAPE is fine if
  it always was; the same number is an incident for a clinic that calibrated
  at 12%.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 60)

from clinic_forecast.data import generate_network_data

data_path = PROJECT_ROOT / "data" / "processed" / "clinic_daily_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
else:
    usage = generate_network_data().usage
    usage["date"] = pd.to_datetime(usage["date"])

## Setting the scene: a "production" month

We treat the final 28 days as the monitored production window. The model is
trained on everything before it (exactly what the batch pipeline would have
done), and the preceding eight weeks serve as the stable reference window for
distribution-shift checks.

In [2]:
from clinic_forecast.models.global_ml import GlobalMLForecaster

origin = usage["date"].max() - pd.Timedelta(days=28)
train = usage[usage["date"] <= origin]
monitored = usage[usage["date"] > origin]
reference = usage[
    (usage["date"] > origin - pd.Timedelta(days=56)) & (usage["date"] <= origin)
]

model = GlobalMLForecaster().fit(train)
combined = pd.concat([train, monitored], ignore_index=True).sort_values(["clinic_id", "date"])
predictions = model.predict_known_future(combined)
scored = monitored.merge(
    predictions[predictions["date"] > origin][["clinic_id", "date", "forecast"]],
    on=["clinic_id", "date"], how="inner",
)

# Training-time quality per clinic = the degradation baseline.
from clinic_forecast.evaluation import evaluate_forecasts

train_tail = train[train["date"] > origin - pd.Timedelta(days=56)]
tail_pred = predictions[
    (predictions["date"] > origin - pd.Timedelta(days=56)) & (predictions["date"] <= origin)
]
tail_scored = train_tail.merge(
    tail_pred[["clinic_id", "date", "forecast"]], on=["clinic_id", "date"], how="inner"
).assign(model="global_ml_hgb")
reference_wape = (
    evaluate_forecasts(tail_scored, group_cols=["clinic_id"])
    .set_index("clinic_id")["wape"].to_dict()
)

## The monitoring report on a healthy month

In [3]:
from clinic_forecast.monitoring import load_monitoring_config, monitoring_report

thresholds = load_monitoring_config(PROJECT_ROOT / "configs" / "monitoring.yaml")
open_monitored = monitored[monitored["is_open"] == 1]
open_reference = reference[reference["is_open"] == 1]

report = monitoring_report(
    scored=scored,
    recent=open_monitored,
    reference=open_reference,
    thresholds=thresholds,
    reference_wape=reference_wape,
)
print(f"{int(report['alert'].sum())} alert(s) out of {len(report)} checks")
report.head(12)

10 alert(s) out of 76 checks


,level,group,check,value,threshold,alert
0,clinic,CLINIC_004,volume_shift_ratio,0.269,0.25,True
1,clinic,CLINIC_005,abs_bias_pct,10.880,10.00,True
2,clinic,CLINIC_005,utilization_shift,0.159,0.15,True
3,clinic,CLINIC_005,volume_shift_ratio,0.531,0.25,True
4,clinic,CLINIC_005,wape_pct,36.072,35.00,True
5,clinic,CLINIC_008,volume_shift_ratio,0.435,0.25,True
6,clinic,CLINIC_008,wape_degradation_ratio,1.361,1.30,True
7,clinic,CLINIC_008,wape_pct,37.230,35.00,True
8,clinic,CLINIC_009,abs_bias_pct,12.422,10.00,True
9,clinic,CLINIC_009,wape_degradation_ratio,1.557,1.30,True


Most checks pass on a normal month; the table surfaces the few that do not
(demand episodes can legitimately trip a volume or degradation check — an
alert means "look", not "the model is broken").

## What each alert means operationally

| Check | What it detects | First action |
| --- | --- | --- |
| `abs_bias_pct` | Systematic over/under-forecast for a clinic or region | If persistent: retrain; staffing has been skewed one direction |
| `wape_pct` | Accuracy below decision-grade in absolute terms | Manual review of that clinic's rosters until resolved |
| `wape_degradation_ratio` | Accuracy materially worse than the model demonstrated at training time | Retraining candidate; check for demand regime change |
| `volume_shift_ratio` | Demand level moved vs reference window | Verify real (new competitor? service change?) - retrain + recalibrate intervals |
| `spend_shift_ratio` | Marketing behaviour changed | Update the marketing plan assumption in the batch pipeline |
| `utilization_shift` | Clinics running hotter/colder against capacity | Rising: more censored peaks, demand target increasingly understates true demand |

## Drill: a demand regime change

To verify alerts actually fire, we simulate a 50% demand surge at two clinics
in the monitored window (the kind of shift a new referral contract or a
closed competitor produces) and re-run the *same* report.

In [4]:
shifted = monitored.copy()
surge_clinics = ["CLINIC_001", "CLINIC_004"]
mask = shifted["clinic_id"].isin(surge_clinics)
shifted.loc[mask, "visits"] = (shifted.loc[mask, "visits"] * 1.5).round().astype(int)

shifted_scored = shifted.merge(
    predictions[predictions["date"] > origin][["clinic_id", "date", "forecast"]],
    on=["clinic_id", "date"], how="inner",
)
drill = monitoring_report(
    scored=shifted_scored,
    recent=shifted[shifted["is_open"] == 1],
    reference=open_reference,
    thresholds=thresholds,
    reference_wape=reference_wape,
)
drill[drill["alert"] & drill["group"].isin(surge_clinics + ["north", "west"])]

,level,group,check,value,threshold,alert
0,clinic,CLINIC_001,abs_bias_pct,37.548,10.00,True
1,clinic,CLINIC_001,volume_shift_ratio,0.816,0.25,True
2,clinic,CLINIC_001,wape_degradation_ratio,1.974,1.30,True
3,clinic,CLINIC_001,wape_pct,40.162,35.00,True
4,clinic,CLINIC_004,abs_bias_pct,36.287,10.00,True
5,clinic,CLINIC_004,volume_shift_ratio,0.902,0.25,True
6,clinic,CLINIC_004,wape_degradation_ratio,1.732,1.30,True
7,clinic,CLINIC_004,wape_pct,37.500,35.00,True
17,region,north,abs_bias_pct,19.746,10.00,True
18,region,west,abs_bias_pct,12.312,10.00,True


The surged clinics light up across multiple checks at once — bias (the model
now under-forecasts them), WAPE degradation, and volume shift — while
unaffected clinics stay quiet. Multi-check agreement on the same clinic is
the strongest retraining signal this layer produces.

## Retraining policy

A reasonable starting policy, refined with experience:

1. **Volume shift + degradation on the same clinic** → retrain now, and
   recalibrate conformal intervals (their residuals are stale by definition).
2. **Bias alert alone, persistent across two windows** → retrain at the next
   scheduled cycle; flag rosters for manual review meanwhile.
3. **Spend shift alone** → fix the input (marketing plan), not the model.
4. **Single-window WAPE alert with no input shift** → likely a demand
   episode; watch, don't churn the model on noise.

The registry (`outputs/model_registry/`) stores calibration-time metrics with
every batch run, so the degradation baseline is always available without
re-deriving it.